<img src="./ccsf.png" alt="CCSF Logo" width=200px style="margin:0px -5px">

# Project 2: Final Project

In this project, you will build a `k`-NN classifier that predicts whether a movie is a thriller or a comedy by analyzing word frequencies from movie scripts. You will clean and prepare the data, select meaningful features, use the `k`-NN algorithm to create a predictive model with training data, fine-tune the model with a validation set, and evaluate it on a test set.

---

## 🎗️ Assignment Reminders

As you work on this project, consider the following:
- **Initialize Otter.** 🚨 Make sure to run the code cell at the top of this notebook that starts with `# Initialize Otter` to load the auto-grader.
- **Complete the Tasks.** Your Tasks are categorized as auto-graded (📍) and manually graded (📍🔎).
    - For all the auto-graded tasks:
        - Replace the `...` in the provided code cell with your own code.
        - Run the `grader.check` code cell to run some tests on your code.
        - Keep in mind that for homework and project assignments, sometimes there are hidden tests that you will not be able to see the results of that we use for scoring the correctness of your response. **Passing the auto-grader does not guarantee that your answer is correct.**
    - For all the manually graded tasks:
        - You might need to provide your own response to the provided prompt. Do so by replacing the template text "_Type your answer here, replacing this text._" with your own stems.
        - You might need to produce a graphic or something else using code. Do so by replacing the `...` in the code cell to generate the image, table, etc.
        - In either case, [review the rubric](https://community.canvaslms.com/t5/Student-Guide/How-do-I-view-the-rubric-for-my-assignment/ta-p/275) on the associated <a href="https://ccsf.instructure.com" target="_blank">Canvas</a> Assignment page to understand the scoring criteria.
- **Code Sharing.** By submitting this project, you agree that you will not share your code directly with anybody but your partner. You are welcome to discuss questions with others but don't share the answers directly. The experience of solving the problems in this project will prepare you for our exams and potentially for future work in this field. If someone asks you for the answer, resist! Instead, you can demonstrate how you would solve a similar problem or you can focus on a specific part of a task.
- **Support.** You are not alone! Review the [Course Support page](https://ccsf-math-108.github.io/materials-fa25/resources/course-support.html) for how to get support in this course. If you're ever feeling overwhelmed or don't know how to make progress, talk to your instructor or a tutor.
- **Advice.** Develop your answers incrementally. To perform a complicated table manipulation, break it up into steps, perform each step on a different line, give a new name to each result, and check that each intermediate result is what you expect. You can add any additional names or functions you want to the provided cells. Make sure that you are using distinct and meaningful variable names throughout the notebook.
- **Variable Names.** Throughout this assignment and all future ones, please be sure to not re-assign variables throughout the notebook! _For example, if you use `max_temperature` in your answer to one question, do not reassign it later on. Otherwise, you will fail tests that you thought you were passing previously!_
- **Re-Submitting.** You may [submit](#🏁-Submit-Your-Assignment-to-Canvas) this assignment as many times as you want before the deadline. Your instructor will score the last version you submit once the deadline has passed.


---

## Set Up Notebook

### Task 00 📍

<!-- BEGIN QUESTION -->

Run the following code cells to set up the notebook.

_You are not responsible for understanding what this code does._

_Points:_ 0

In [ ]:
# Initialize Otter
import otter
grader = otter.Notebook("project2.ipynb")

In [ ]:
# Configure this Notebook
import datascience
import numpy as np
from datascience import *
import statsmodels
import matplotlib
%matplotlib inline
import matplotlib.pyplot as plt
plt.style.use('fivethirtyeight')
import plotly.express as px
import warnings
warnings.simplefilter(action='ignore', category=UserWarning)

<!-- END QUESTION -->

---

## 💾 Part 1: Data Processing

---

### Section 1.1: Stem Proportions

---

#### Accessing the Data

You will work with a collection of movie screenplays and attempt to predict each movie's genre based on its dialogue. The [Cornell Movie-Dialogs Corpus](https://www.cs.cornell.edu/~cristian/Cornell_Movie-Dialogs_Corpus.html) contains a large collection of fictional conversations extracted from raw movie scripts. The `movies.csv` file records the proportion of 5,000 common stems within each movie based on this corpus.

Run the cell below to read the `movies` table. **It may take up to a minute to load.**

In [ ]:
movies = Table.read_table('movies.csv')
movies_by_title = movies.index_by('Title') # Used later
print('stems with frequencies:', movies.drop(np.arange(5)).num_columns) 
print('Movies with genres:', movies.num_rows)
movies

---

#### Bag-of-stems

A **bag-of-stems representation** is a numerical model of text that captures only the (relative) frequencies of individual stems. It is commonly used in [natural language processing](https://en.wikipedia.org/wiki/Natural_language_processing). While this approach discards information such as word order, context, speakers, and character details, it retains enough content in a compact format to serve as a useful starting point for machine learning applications.

---

#### Defining Proportions

Here is one row of the table and some of the proportions of stems that were said in the movie.

In [ ]:
movies.where("Title", "runaway bride").select(0, 1, 2, 3, 4, 14, 49, 1042, 4004)

The cell above shows a few columns from the row for the comedy *Runaway Bride*. The conversations analyzed in this movie include 4,895 total stems. The word **"it"** appears 115 times, which is its frequency. Because different movies have different total word counts, it’s more meaningful to compare relative frequencies. For example, $115 / 4895 \approx 0.0234092$, so the relative frequency (or proportion) for **"it"** is `0.0234092` for *Runaway Bride*. The word **“england”** does not appear in the script, so its relative frequency is `0`.

---

#### Quickly Accessing Proportions

All the movie titles in `movies` are unique and presented in lower-case format. Since this dataset is large, it can take a long time to access information in the way that you are used to in the course. The following `row_for_title` function provides fast access to the one row for each title.

Run the following cell to define that function and quickly find the relative frequency of "fun" in the movie *Toy Story*.

In [ ]:
def row_for_title(title):
    """Return the row for a title, similar to the following expression (but faster)
    
    movies.where('Title', title).row(0)
    """
    return movies_by_title.get(title)[0]

# Call the function
row_for_title('toy story').item('fun') 

Notice that `row_for_title('toy story')` returns the row object for that movie and `.item('fun')` extracted the relative frequency for the word `fun`.

---

#### Task 1.1 📍

Using the `row_for_title` function, assign the frequency for the word `'back'` in `'the terminator'` to `ill_be_back`.

_Points:_ 2

In [ ]:
ill_be_back = ...
ill_be_back

In [ ]:
grader.check("task_1_1")

---

### Section 1.2: Stem Stemming

The columns in the `movies` table, aside from "Title", "Year", "Rating", "Genre", and "# stems", represent stems that appear in conversations from the movies in our dataset. These stems have been *stemmed*, meaning they have been heuristically shortened so that different [inflected](https://en.wikipedia.org/wiki/Inflection) forms of the same base word correspond to the same string. For example, there is a column `"manag"`. This is not an English word, but it groups together the proportions of stems like "manage", "manager", "managed", "managerial", and possibly others, for each movie. This approach is common in machine learning and natural language processing.

Stemming can make it harder to look up specific stems, so we also provide a table named `vocab_table`, which lists examples of unstemmed stems associated with each stemmed form.

Run the code below to load that table.

In [ ]:
stemmed = movies.labels[3:]

vocab_table = (
    Table()
    .with_column('Stem', stemmed)
    .join('Stem', Table.read_table('stem.csv'))
)

vocab_table.take(np.arange(4900, 4910))

The next few tasks will help you explore the stemmed stems in this dataset.

---

#### Task 1.2.1 📍

Using `vocab_table`, find the stemmed version of the word "homicide" and assign the value to `stemmed_version`.


_Points:_ 3

In [ ]:
stemmed_version = ...
stemmed_version

In [ ]:
grader.check("task_1_2_1")

---

#### Task 1.2.2 📍

What stem in the dataset has the most stems that are associated with it? Assign `most_stem` to that stem.

_Points:_ 2

In [ ]:
most_stem = ...
most_stem

In [ ]:
grader.check("task_1_2_2")

---

#### Task 1.2.3 📍

Find the longest word(s) in the dataset that were **not shortened to their stem** and assign them to `longest_unstemmed` as an array.  

**Hints:**

- The `vocab_table` has two columns: one for stems and one for unstemmed (normal) words. Words whose length is equal to their stem are **not shortened**.  
- Use a table function that applies a function to every element in a column. Check the [Quick Reference](https://ccsf-math-108.github.io/materials-fa25/resources/quick-reference.html) if you are unsure which one to use.  
- A helpful strategy is to add columns for **word length** and **stem length**, then create a column for their **difference**. stems that were not shortened will have a difference of 0.  
- You may need to sort or filter the table to find all stems with the maximum length.

_Points:_ 2

In [ ]:
longest_unstemmed = ...
longest_unstemmed

In [ ]:
grader.check("task_1_2_3")

---

#### Task 1.2.4 📍

How many stems in the dataset correspond to **only one word**?  

For example, if the stem `"book"` maps only to the word `"books"`, and the stem `"a"` maps only to `"a"`, both should be counted as stems that map to a single word.  

Assign the variable `count_single_stems` to the total number of stems that map to exactly one word.

_Points:_ 2

In [ ]:
count_single_stems = ...
count_single_stems

In [ ]:
grader.check("task_1_2_4")

---

### Section 1.3: Splitting the dataset

We will use the `movies` dataset for three tasks:

1. **Training** our movie genre classifiers  
2. **Selecting the best value of** \(k\) using a validation set  
3. **Evaluating final performance** on a test set  

To support these steps, we need three separate datasets: a **training set**, a **validation set**, and a **test set**.  

The goal of a classifier is to make predictions on new data that resemble the training examples. To assess its performance, we compare its predictions to the actual genres of movies it has never seen. This requires that the training, validation, and test sets contain no overlapping movies. Since the dataset has already been randomly permuted, we can create non-overlapping subsets by slicing the table.  

We will use the first 70% of the dataset for training, the next 15% for validation, and the remaining 15% for testing. 

Run the following code to perform this split and create the tables `train`, `validation`, and `test`.

In [ ]:
# Define proportions for training, validation, and test sets
training_proportion = 0.70
validation_proportion = 0.15
test_proportion = 0.15  # Remaining portion

num_movies = movies.num_rows
num_train = int(num_movies * training_proportion)
num_validation = int(num_movies * validation_proportion)
num_test = num_movies - num_train - num_validation

# Split the dataset
np.random.seed(1234) #Needed for scoring consistency
movies = movies.sample(with_replacement=False)
train = movies.take(np.arange(num_train))
validation = movies.take(np.arange(num_train, num_train + num_validation))
test = movies.take(np.arange(num_train + num_validation, num_movies))

print("Training: ",   train.num_rows, ";",
      "Validation: ", validation.num_rows, ";",
      "Test: ",       test.num_rows)

---

#### Task 1.3 📍🔎

<!-- BEGIN QUESTION -->

Draw a horizontal bar chart with three bars that show the proportion of Comedy movies in each dataset (`train`, `validation`, and `test`). The three bars should be labeled "Training", "Validation", and "Test". Complete the function `comedy_proportion` first; it should help you create the bar chart. 

**Note**: Refer to [Section 7.1](https://ccsf-math-108.github.io/textbook/chapters/07/1/Visualizing_Categorical_Distributions.html#bar-chart) of the textbook if you need a refresher on bar charts.

_Points:_ 2

In [ ]:
def comedy_proportion(table):
    # Return the proportion of movies in a table that have the comedy genre.
    ...
    return ...

# The staff solution took multiple lines.  Start by creating a table.
# If you get stuck, think about what sort of table you need for barh to work
...

<!-- END QUESTION -->

---

## 🏘️ Part 2: A Guided Walkthrough of Classification

---

### Section 2.1: Classifying a movie

---

#### k-Nearest Neighbors

[k-Nearest Neighbors (k-NN)](https://ccsf-math-108.github.io/textbook/chapters/17/1/Nearest_Neighbors.html) is a classification algorithm.  
* Given some numerical *attributes* (also called *features*) of an unseen example, it determines which category the example belongs to by comparing it to previously seen examples.  
* Predicting the category of an example is called *labeling*, and the predicted category is the *label*.

In this project, one attribute you have for each movie is the proportion of times a particular word appears in the selected conversations from the movie. The labels are the two genres: comedy and thriller. The algorithm needs many examples for which both the attributes and labels are known, and these are provided in the `train` table.

To build intuition, we will have you will start by visualizing the algorithm rather than only describing it.

---

#### Euclidean Distance

In k-NN, we classify a movie by finding the `k` movies in the *training set* that are most similar according to the features we choose. These similar movies are called the *nearest neighbors*. The algorithm assigns the movie to the most common category among its `k` nearest neighbors.

For now, we will use only two features so that we can plot each movie. The features we will use are the proportions of the stems **“water”** and **“feel”** in the movie’s conversations. For the movie *Monty Python and the Holy Grail*, 0.000804074 of its stems are “water” and 0.0010721 are “feel”. Let's imagine that its genre is unknown.

To compare movies, we need a precise notion of similarity. We will measure the *distance* between two movies using the straight-line distance between their feature points on a scatter plot.

**This distance is called the Euclidean distance, with formula**  

$$
\sqrt{(x_1 - x_2)^2 + (y_1 - y_2)^2}.
$$

---

#### Classifying Example

For example, in *Clerks.*, 0.00016293 of its stems are “water” and 0.00154786 are “feel”. Its distance from *Monty Python and the Holy Grail* based on these two features is  

$$
\sqrt{(0.000804074 - 0.000162933)^2 + (0.0010721 - 0.00154786)^2} \approx 0.000798379.
$$

Another training movie, *The Godfather*, has 0 “water” and 0.00015122 “feel”.

The function below plots the “water” and “feel” features for these movies. As you can see in the resulting plot, *Monty Python and the Holy Grail* is closer to *Clerks.* than to *The Godfather* based on these features, which makes sense since both are comedies, while *The Godfather* is a thriller.

In [ ]:
# Just run this cell.
def plot_with_two_features(unlabeled_movie, training_movies, x_feature, y_feature):
    """Plot a test movie and training movies using two features."""
    unlabeled_movie_row= row_for_title(unlabeled_movie)
    distances = Table().with_columns(
            x_feature, [unlabeled_movie_row.item(x_feature)],
            y_feature, [unlabeled_movie_row.item(y_feature)],
            'Color',   ['unknown'],
            'Title',   [unlabeled_movie]
        )
    for movie in training_movies:
        row = row_for_title(movie)
        distances.append([row.item(x_feature), row.item(y_feature), row.item('Genre'), movie])
    distances.scatter(x_feature, y_feature, group='Color', labels='Title', s=50)
    
training = ["clerks.", "the godfather"] 
plot_with_two_features("monty python and the holy grail", training, "water", "feel")
plt.axis([-0.0008, 0.001, -0.004, 0.007]);

---

#### Task 2.1.1 📍

Compute the Euclidean distance (defined in the section above) between the two movies, *Monty Python and the Holy Grail* and *The Godfather*, using the `water` and `feel` features only.  Assign the distance to the name `one_distance`. 

**Hints:**
1. If you have a row, you can use `item` to get a value from a column by its name.  For example, if `r` is a row, then `r.item("Genre")` is the value in column `"Genre"` in row `r`.
2. Refer to the beginning of Part 1 if you don't remember what `row_for_title` does.
3. In the formula for Euclidean distance, think carefully about what `x` and `y` represent. Refer to the example in the text above if you are unsure.

_Points:_ 2

In [ ]:
python = row_for_title("monty python and the holy grail") 
godfather = row_for_title("the godfather") 
python_x = python.item("water")
python_y = python.item("feel")
godfather_x = godfather.item("water")
godfather_y = godfather.item("feel")

one_distance = ...
one_distance

In [ ]:
grader.check("task_2_1_1")

---

Below, we've added a third movie, *The Silence of the Lambs*. Before, the point closest to *Monty Python and the Holy Grail* was *Clerks.*, a comedy movie. However, now the closest point to *The Godfather* is *The Silence of the Lambs*, a thriller movie.

In [ ]:
training = ["clerks.", "the godfather", "the silence of the lambs"] 
plot_with_two_features("monty python and the holy grail", training, "water", "feel") 
plt.axis([-0.0008, 0.001, -0.004, 0.007]);

---

#### Task 2.1.2 📍

Complete the function `distance_two_features` that computes the Euclidean distance between any two movies, using two features. The last two lines call your function to show that *Monty Python and the Holy Grail* is closer to *The Silence of the Lambs* than it is to *Clerks*. 

_Points:_ 2

In [ ]:
def distance_two_features(title0, title1, x_feature, y_feature):
    '''Compute the distance between two movies with titles title0 and title1.
    
    Only the features named x_feature and y_feature are used when computing the distance.
    '''
    ...

for movie in make_array("clerks.", "the silence of the lambs"):
    movie_distance = distance_two_features(movie, "monty python and the holy grail", "water", "feel")
    print(movie, 'distance:\t', movie_distance)

In [ ]:
grader.check("task_2_1_2")

---

#### Task 2.1.3 📍

Define the function `distance_from_python` so that it works as described in its documentation. 

**Note:** Your solution should not use arithmetic operations directly. Instead, it should make use of existing functionality above!

_Points:_ 2

In [ ]:
def distance_from_python(title):
    """Returns the distance between the provided movie title and "monty python and the holy grail", 
    based on the features "water" and "feel".
    
    This function takes a single argument:
      title: A string, the name of a movie.
    """
    
    ...

# Calculate the distance between "Clerks." and "Monty Python and the Holy Grail"
distance_from_python('clerks.')

In [ ]:
grader.check("task_2_1_3")

---

#### Task 2.1.4 📍

Using the features `"water"` and `"feel"`, find the names and genres of the five movies in the **training set** that are closest to *Monty Python and the Holy Grail*. Create a **table** named `close_movies` that contains those five movies with the columns `"Title"`, `"Genre"`, `"water"`, and `"feel"`, along with a column `"distance from python"` showing each movie's distance from *Monty Python and the Holy Grail*. The table should be **sorted in ascending order by `"distance from python"`**.

**Note:** Make sure that *Monty Python and the Holy Grail* is NOT included as one of the five closest movies to itself.

**Hint:** Your final table should contain only five rows. How can you select the first five rows of a table?


_Points:_ 4

In [ ]:
...
close_movies = ...
close_movies

In [ ]:
grader.check("task_2_1_4")

---

#### Task 2.1.5 📍

Next, clasify *Monty Python and the Holy Grail* based on the genres of the closest movies. 

To do so, define the function `most_common` so that it works as described in its documentation below. 

_Points:_ 1

In [ ]:
def most_common(label, table):
    """The most common element in a column of a table.
    
    This function takes two arguments:
      label: The label of a column, a string.
      table: A table.
     
    It returns the most common value in the label column of the table.
    In case of a tie, it returns any one of the most common values.    
    """
    ...

# Calling most_common on your table of 5 nearest neighbors classifies
# "monty python and the holy grail" as a comedy movie, 3 votes to 2. 
most_common('Genre', close_movies)

In [ ]:
grader.check("task_2_1_5")

---

#### More than Two Features

Euclidean distance still works when we use more than two features. For `n` features, we compute the difference between the two movies on each feature, square each of the `n` differences, add them together, and take the square root of the sum. Next, we will have you extend the classification process from the previous part to consider more than two features at once. 

---

#### Task 2.1.6 📍

Write a function `distance` that computes the Euclidean distance between two **arrays** of **numerical** features (for example, arrays of word proportions). The function should work for arrays of any length, as long as both arrays have the same length.

After defining the function, use it to compute the distance **between the first and second movies** in the **training set**, using **all feature columns**. Remember that the first five columns of the table are not features.

**Hints:**
1. To turn a table row into an array, use `np.array`. For example, if `t` is a table, then `np.array(t.row(0))` converts row 0 into an array. The optional variables `array_values_movie_1` and `array_values_movie_2` are there to help you build two feature arrays to test your function.
2. Before computing `distance_first_to_second`, make sure to drop the first five columns of the training set, since only the remaining columns contain the word proportion features.


_Points:_ 7

In [ ]:
def distance(features_array1, features_array2):
    """The Euclidean distance between two arrays of feature values."""
    ...

train_features_only = ...
array_values_movie_1 = ...
array_values_movie_2 = ...
distance_first_to_second = ...
distance_first_to_second

In [ ]:
grader.check("task_2_1_6")

---

#### Fast Distance

As before, our goal is to find the movies in the training set that are most similar to an unlabeled movie. That means computing the Euclidean distance from the validation movie (using `my_features`) to every movie in the training set, which could be a large set. You *could* do this with a `for` loop using the `distance` function you just wrote, but that approach gets slow as the size of the training set increases.

To preview where we're headed, we’ve included a more efficient helper called `fast_distances` that carries out the same task much more quickly. Run the following code cell to define `fast_distances`. Take a moment to read through its documentation so you know what it returns. You won’t need to unpack the code inside unless you’re curious.

In [ ]:
# Just run this cell to define fast_distances.

def fast_distances(example_row, train_table):
    """Return an array of the distances between example_row and each row in train_table.

    Takes 2 arguments:
      example_row: A row of a table containing features of one
        example movie (e.g., validation_my_features.row(0)).
      train_table: A table of features (for example, the whole
        table train_my_features)."""
    assert train_table.num_columns < 50, "Make sure you're not using all the features of the movies table."
    assert type(example_row) != datascience.tables.Table, "Make sure you are passing in a row object to fast_distances."
    assert len(example_row) == len(train_table.row(0)), "Make sure the length of example row is the same as the length of a row in train_table."
    counts_matrix = np.asmatrix(train_table.columns).transpose()
    diff = np.tile(np.array(list(example_row)), [counts_matrix.shape[0], 1]) - counts_matrix
    np.random.seed(0) # For tie breaking purposes
    distances = np.squeeze(np.asarray(np.sqrt(np.square(diff).sum(1))))
    eps = np.random.uniform(size=distances.shape)*1e-10 #Noise for tie break
    distances = distances + eps
    return distances

If you try to use the `fast_distances` function with the entire `train` table, you may get an error. This happens because processing all of the training data at once could exceed the available memory and crash the kernel.

---

## ✨ Part 3: Feature Engineering

---

### Section 3.1: Correlation

---

#### Average Occurrence

Unfortunately, using all of the features has some drawbacks. One important issue is the loss of *computational efficiency*. Computing Euclidean distances becomes slow when there are many features. You may have already noticed this in the previous task.

To address this issue, we will guide you in selecting only 10 features. Our goal is to have you choose features that are highly *discriminative*, meaning they help the classifier correctly label as often as possible. Selecting features that improve classifier performance is known as *feature selection*, and more broadly, *feature engineering*.

Run the following cell to load a table showing the average occurrence of each feature in each genre. This table will help you identify the most discriminative features to use for classification.

In [ ]:
stems_with_averages = Table().read_table('stem_averages.csv')
stems_with_averages

---

#### Visualizing the Relationship

In [ ]:
stems_with_averages.scatter('Average Occurrence (Comedy)', 'Average Occurrence (Thriller)')
plt.title('Average Occurrences by Genre')
plt.show()

This relationship appears to be positive and follows a strong linear pattern.

---

#### Task 3.1.1 📍

What is the correlation coefficient between the Average Occurrence (Comedy) and Average Occurrence (Thriller) variables? Assign that correlation value (float) to `r`.

_Points:_ 2

In [ ]:
r = ...
r

In [ ]:
grader.check("task_3_1_1")

---

Run the following code to create an interactive scatter plot. Each word is represented as a dot, with its position determined by the average occurrence in comedy movies (horizontal axis) and in thriller movies (vertical axis).

**Notes:**
1. The line shown on the plot is the linear regression line, not the line $y = x$.  
2. The plot uses the interactive library Plotly Express. Hover over points to see the word each dot represents, and use the toolbar to zoom, pan, or explore different parts of the graph.

In [ ]:
stems_with_averages_df = stems_with_averages.to_df()
fig = px.scatter(stems_with_averages_df, 
                 x="Average Occurrence (Comedy)", 
                 y="Average Occurrence (Thriller)", 
                 hover_data=['Stem'], 
                 trendline="ols",
                 trendline_color_override="pink",
                 title="Average Occurrences by Genre")
fig.update_layout(autosize=False, width=600, height=600)
fig.show()

---

#### Interpreting the Graph

The following tasks will ask you to interpret the plot above.

---

#### Task 3.1.2 📍

What properties do stems in the bottom left corner have? Assign one of the following choices (integer) to `bottom_left`:

1. The word is common in both comedy and thriller movies 
2. The word is uncommon in comedy movies and common in thriller movies
3. The word is common in comedy movies and uncommon in thriller movies
4. The word is uncommon in both comedy and thriller movies
5. It is not possible to say from the plot 

_Points:_ 1

In [ ]:
bottom_left = ...

In [ ]:
grader.check("task_3_1_2")

---

#### Task 3.1.3 📍

What properties do stems in the top left corner have? Assign one of the following choices (integer) to `top_left`:

1. The word is common in both comedy and thriller movies 
2. The word is uncommon in comedy movies and common in thriller movies
3. The word is common in comedy movies and uncommon in thriller movies
4. The word is uncommon in both comedy and thriller movies
5. It is not possible to say from the plot 

_Points:_ 1

In [ ]:
top_left = ...

In [ ]:
grader.check("task_3_1_3")

---

#### Task 3.1.4 📍

What properties do stems in the bottom right corner have? Assign one of the following choices (integer) to `bottom_right`:

1. The word is common in both comedy and thriller movies 
2. The word is uncommon in comedy movies and common in thriller movies
3. The word is common in comedy movies and uncommon in thriller movies
4. The word is uncommon in both comedy and thriller movies
5. It is not possible to say from the plot 

_Points:_ 1

In [ ]:
bottom_right = ...

In [ ]:
grader.check("task_3_1_4")

---

#### Task 3.1.5 📍

What properties do stems in the top right corner have? Assign one of the following choices (integer) to `top_right`:

1. The word is common in both comedy and thriller movies 
2. The word is uncommon in comedy movies and common in thriller movies
3. The word is common in comedy movies and uncommon in thriller movies
4. The word is uncommon in both comedy and thriller movies
5. It is not possible to say from the plot 

_Points:_ 1

In [ ]:
top_right = ...

In [ ]:
grader.check("task_3_1_5")

---

#### Difference in Average Occurrences

You might think that a word is highly discriminative if its average occurrence in comedy movies is very different from its average occurrence in thriller movies. This is a good starting point, but there are some important nuances to consider, which we will explore next.

---

#### Task 3.1.6 📍

Determine the top 10 stems based on the largest difference in average occurrence for comedy and average occurrence for thriller movies. Assign those stems as an array to `big_diffs`.

_Points:_ 3

In [ ]:
big_diffs = ...
big_diffs

In [ ]:
grader.check("task_3_1_6")

You should notice that these stems are pretty familiar and may not be the most helpful in distinguishing between a comedy and a thriller. Next, we will have you explore this idea further.

---

#### An Issue

If you just pick the stems with the largest absolute difference in proportions, you’ll often get extremely common stems like “and”, “the”, “of”, etc., because even small proportional differences in very frequent stems produce large absolute differences. So, unfortunately, the stems in `big_diffs` aren’t discriminative for identifying a thriller or comedy genre, in general.

---

#### Task 3.1.7 📍

Assign the `bool` value (`True` or `False`) which best describes the following statement to `ecological_correlation`: 

"Assuming that the linear relationship observed between the **average occurrence of stems in comedies vs. thrillers** also holds for the **occurrence of those stems in each individual movie** is an example of an ecological correlation."

_Points:_ 2

In [ ]:
ecological_correlation = ...
ecological_correlation

In [ ]:
grader.check("task_3_1_7")

---

### Section 3.2: Identifying the General Pattern

---

#### Overfitting

The stem `'Freddi'` corresponds to the character Freddy Krueger in the *A Nightmare on Elm Street* movies. Its occurrence is strongly associated with that specific movie, but it’s very rare in other films. While it could help identify that one thriller, it doesn’t generalize to other movies, making it an example of a machine learning issue called overfitting.

---

#### Standardizing

Avoiding overfitting is all about making sure your model learns general patterns rather than focusing on special cases in your training data.
* We need to use stems that occur enough across a particular genre so that we can make sure the pattern generalizes.
* We need to avoid stems that are too general (and likely too frequent).
* The standardization of the data helps ensure that very common stems like `'the'` don’t dominate the Euclidean distance calcluation.

---

#### Task 3.2 📍

1. Standardize the average occurrences for both genres in `stems_with_averages`.
2. Add these standardized values as new columns to form a table called `stems_with_averages_su`. It should contain the columns `'Stem'`, `'Average Occurrence (Comedy)'`, `'Average Occurrence (Thriller)'`, `'Average Occurrence (Comedy) (Standardized)'`, and `'Average Occurrence (Thriller) (Standardized)'`.
3. Filter `stems_with_averages_su` to keep only the words whose standardized averages for **both** genres fall within 3 standard deviations of the mean. Assign this filtered table to `typical_words_su`.

_Points:_ 3

In [ ]:
stems_with_averages_su = ...
typical_stems_su = ...

display(typical_stems_su)

# Create an interactive scatterplot
stems_with_averages_su_df = typical_stems_su.to_df()
fig = px.scatter(stems_with_averages_su_df, 
                 x="Average Occurrence (Comedy) (Standardized)", 
                 y="Average Occurrence (Thriller) (Standardized)", 
                 hover_data=['Stem'], 
                 trendline="ols",
                 trendline_color_override="pink",
                 title="Average Occurence of Stems by Genre")
fig.update_layout(autosize=False, width=600, height=600)
fig.show()

In [ ]:
grader.check("task_3_2")

---

### Section 3.3: Selecting Your Features

---

Now that you've explored how each word behaves across the two genres, you can move on to selecting your features.

#### Task 3.3 📍

Create an array called `my_features` that contains the 10 stems in `typical_stems_su` that are the furthest away from the linear regression line shown in Task 3.2.

_Points:_ 4

In [ ]:
my_features = ...

display(my_features)

# Select the 10 features of interest from the train, validation, and test sets
train_my_features = train.select(my_features)
validation_my_features = validation.select(my_features)
test_my_features = test.select(my_features)

In [ ]:
grader.check("task_3_3")

---

### Section 3.4: Classifying a Movie

---

#### Task 3.4.1 📍

1. Use the `fast_distances` function (defined earlier) to compute the distance between the first movie in your validation set and every movie in your training set, using the features in `my_features`.
2. Create a new table named `genre_and_distances` with one row per training movie and two columns:
    * `"Genre"` for each training movie
    * `"Distance"` to the first validation movie
3. Sort `genre_and_distances` in ascending order by distance.

_Points:_ 4

In [ ]:
first_validation_row = ...
genre_and_distances = ...
genre_and_distances

In [ ]:
grader.check("task_3_4_1")

---

#### Task 3.4.2 📍

Compute the 5-nearest neighbors classification for the first movie in the validation set. Determine its predicted genre by finding the most common genre among its 5 closest training movies, based on the distances you calculated, and assign it to `my_assigned_genre`.  

Then check whether this predicted genre matches the movie’s actual genre, and assign a Boolean value (`True` or `False`) to `my_assigned_genre_was_correct` to indicate whether the prediction was correct. It’s fine if the classifier gets this example wrong.

**Hints:**
1. Use the `most_common` function defined earlier.  
2. The comparison operator `==` could be useful to check if.


_Points:_ 2

In [ ]:
# Set my_assigned_genre to the most common genre among these.
my_assigned_genre = ...

# Set my_assigned_genre_was_correct to True if my_assigned_genre
# matches the actual genre of the first movie in the validation set, False otherwise.
my_assigned_genre_was_correct = ...

print("The assigned genre, {}, was{}correct.".format(my_assigned_genre, " " if my_assigned_genre_was_correct else " not "))

In [ ]:
grader.check("task_3_4_2")

---

### Section 3.5: Creating a Classifier Function


Now we can write a single function that encapsulates the whole process of classification.

---

#### Task 3.5 📍

Write a function called `classify` that takes four arguments:

* `example_row`: A row of features for a single movie to classify (e.g., `validation_my_features.row(0)`).
* `train_features`: A table containing the same features for all training movies (e.g., `train_my_features`).
* `train_labels`: An array of labels (e.g., `"comedy"` or `"thriller"`) with one label per row of the training table, in the same order.
* `k`: The number of neighbors to use in the classification.

The function should return the class (`'comedy'` or `'thriller'`) predicted by a k-nearest neighbors classifier for the given movie.

_Points:_ 2

In [ ]:
def classify(example_row, train_features, train_labels, k):
    """Return the most common class among k nearest neighbors to example_row."""
    distances = fast_distances(example_row, train_features)
    genre_and_distances = ...
    ...

# Apply this function to the first movie in the validation set.
classify(first_validation_row, train_my_features, train.column('Genre'), 7)

In [ ]:
grader.check("task_3_5")

---

## 🧑‍🔬 Part 4: Classifier Tuning and Model Evaluation

---

### Section 4.1: Fine Tuning

Now that we have a classifier function with a set of features, it's time to fine-tune it. In this case, that means finding an optimal value of `k` to use. One way to do this is to try several different values of `k` on a validation set, compute the proportion of correctly labeled movies in the validation set, and then choose the `k` that results in the highest proportion. This process helps ensure that the classifier performs well on new, unseen movies rather than just memorizing the training set.

---

#### Task 4.1.1 📍🔎

<!-- BEGIN QUESTION -->

Create a graphic showing the proportion of correctly labeled movies in the validation set when using the `classify` function for the first 20 odd values of `k`.  

* The horizontal axis should represent the values of `k`.  
* The vertical axis should represent the proportion of movies in the validation set that were correctly classified for each `k`.

_Points:_ 2

In [ ]:
...

plt.title('Proportion Correct in Validation for Various k Values')
plt.show()

<!-- END QUESTION -->

---

#### An Optimal k

Looking at the graph, you can pick an optimal value of `k` that maximizes the proportion of correctly labeled movies in the validation set. If multiple values of `k` achieve the highest proportion, choose the **smallest** of those values.

---

#### Task 4.1.2 📍

Using the method above, assign the optimal value of `k` (an integer) to `optimal_k`.

_Points:_ 2

In [ ]:
optimal_k = ...
optimal_k

In [ ]:
grader.check("task_4_1_2")

---

### Section 4.2: Evaluating your classifier

#### Accuracy

Now that you've fine-tuned the classifier by selecting an optimal value for `k`, it's time to evaluate it. In this course, we use a single metric called **accuracy** to measure performance. The accuracy of a classifier is defined as the proportion of movies in the test set that are correctly labeled.  

The test set should not have been used or viewed at any point during training or validation in order to follow the spirit of this methodology.

---

#### Task 4.2 📍

Using the test set and the training set, apply the classifier with the selected optimal value of `k` and assign the resulting proportion of correctly labeled movies to the variable `accuracy`.

_Points:_ 2

In [ ]:
accuracy = ...
print(f'The accuracy of your classifer is {accuracy*100: 0.2f}%.')

In [ ]:
grader.check("task_4_2")

---

## 🎉 Reflection

Great work. You’ve completed one full iteration of designing a classifier. Here’s a quick recap of the process:

1. Create training, validation, and test sets.  
2. Choose a classification algorithm.  
3. Identify useful features.  
4. Define a classifier function using those features and the training set.  
5. Fine-tune the classifier with the validation set.  
6. Evaluate performance on the test set.  

Machine learning work usually involves many cycles of refinement. As you move forward, keep revisiting your goals, the quality of your data, and your overall approach to building the classifier.

---

## Final Task 📍

<!-- BEGIN QUESTION -->

You're almost done! Run the cell below to check that you passed all of the non-hidden auto-graded tasks, and then follow the instructions below to submit your assignment to Canvas.

_Points:_ 0

In [ ]:
grader.check_all()

<!-- END QUESTION -->

---

## 🏁 Submit Your Assignment to Canvas

1. **Review the Rubric:** View the rubric on the Canvas assignment page to understand the scoring criteria.  
2. **Review Manually Graded Tasks:** Verify that you have responded to all manually graded tasks marked with 📍🔎.  
3. **Review Auto-Graded Tasks:** Inspect the output from the `grade.check_all()` command above. Ensure you have passed all available tests for the auto-graded tasks in this notebook marked with 📍.  
   - This command executes all auto-grader tests sequentially.  
   - Note that Homework and Project assignments include hidden auto-grader tests that are not visible to you. Passing all visible tests does **not** guarantee that your solution is fully correct.  
4. **Save Your Work:** In the notebook toolbar, go to `File → Save Notebook` to save your work and create a checkpoint.  
5. **Download the Notebook:** In the notebook toolbar, go to `File → Download IPYNB` to download the notebook (`.ipynb`) file.  
6. **Upload to Canvas:** Upload the downloaded `.ipynb` file to the appropriate Canvas assignment page.

---

## Attribution

This content is licensed under the <a href="https://creativecommons.org/licenses/by-nc-sa/4.0/">Creative Commons Attribution-NonCommercial-ShareAlike 4.0 International License (CC BY-NC-SA 4.0)</a> and derived from the <a href="https://www.data8.org/">Data 8: The Foundations of Data Science</a> offered by the University of California, Berkeley.

<img src="./by-nc-sa.png" width=100px>